In [ ]:
import os
import numpy as np
import cv2
import tifffile as tiff
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 4          # safer for 512×512×2
EPOCHS = 50
LR = 1e-4

DATASET_DIR = "/Volumes/Windows8_OS/Dataset/Dataset-OG"

In [ ]:
def read_image(path):
    img = tiff.imread(path)

    # Handle (2,H,W) or (H,W,2)
    if img.ndim == 3 and img.shape[0] == 2:
        vv = img[0]
        vh = img[1]
    else:
        vv = img[..., 0]
        vh = img[..., 1]

    vv = vv.astype(np.float32)
    vh = vh.astype(np.float32)

    # Per-channel normalization
    vv = (vv - vv.min()) / (vv.max() - vv.min() + 1e-8)
    vh = (vh - vh.min()) / (vh.max() - vh.min() + 1e-8)

    vv = cv2.resize(vv, (IMG_SIZE, IMG_SIZE))
    vh = cv2.resize(vh, (IMG_SIZE, IMG_SIZE))

    img = np.stack([vv, vh], axis=-1)  # (512,512,2)
    return img

In [ ]:
def read_mask(path):
    mask = tiff.imread(path)

    if mask.ndim > 2:
        mask = mask[..., 0]

    mask = (mask > 0).astype(np.float32)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    return mask[..., None]   # (512,512,1)

In [ ]:
def load_paths(root):
    imgs, masks = [], []

    for cls in ["Oil", "No_Oil"]:
        img_dir = os.path.join(root, "Images", cls)
        mask_dir = os.path.join(root, "Mask", cls)

        for img_name in os.listdir(img_dir):
            base, ext = os.path.splitext(img_name)

            img_path = os.path.join(img_dir, img_name)
            mask_path = os.path.join(mask_dir, base + "_segmentation" + ext)

            if not os.path.exists(mask_path):
                continue

            imgs.append(img_path)
            masks.append(mask_path)

    return imgs, masks

In [ ]:
imgs, masks = load_paths(DATASET_DIR)

train_i, val_i, train_m, val_m = train_test_split(
    imgs, masks, test_size=0.2, random_state=42
)

In [ ]:
def generator(imgs, masks):
    for i, m in zip(imgs, masks):
        try:
            yield read_image(i), read_mask(m)
        except GeneratorExit:
            return
        except Exception as e:
            print("Skipping file:", i)
            continue

In [ ]:
def make_dataset(imgs, masks, train=True):
    ds = tf.data.Dataset.from_generator(
        lambda: generator(imgs, masks),
        output_signature=(
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 2), tf.float32),
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 1), tf.float32)
        )
    )
    if train:
        ds = ds.shuffle(50)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
train_ds = make_dataset(train_i, train_m, True)
val_ds   = make_dataset(val_i, val_m, False)

In [ ]:
def conv_block(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
    x = tf.keras.layers.ReLU()(x)
    return x


def build_unet2d():
    inputs = tf.keras.layers.Input((512, 512, 2))

    # -------- Encoder (Contraction Path) --------
    c1 = conv_block(inputs, 16)
    p1 = tf.keras.layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 32)
    p2 = tf.keras.layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 64)
    p3 = tf.keras.layers.MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 128)
    p4 = tf.keras.layers.MaxPooling2D((2, 2))(c4)

    c5 = conv_block(p4, 256)
    p5 = tf.keras.layers.MaxPooling2D((2, 2))(c5)

    c6 = conv_block(p5, 512)
    p6 = tf.keras.layers.MaxPooling2D((2, 2))(c6)

    # -------- Bottleneck --------
    bn = conv_block(p6, 1024)

    # -------- Decoder (Expansion Path) --------
    u1 = tf.keras.layers.UpSampling2D((2, 2))(bn)
    u1 = tf.keras.layers.Concatenate()([u1, c6])
    c7 = conv_block(u1, 512)

    u2 = tf.keras.layers.UpSampling2D((2, 2))(c7)
    u2 = tf.keras.layers.Concatenate()([u2, c5])
    c8 = conv_block(u2, 256)

    u3 = tf.keras.layers.UpSampling2D((2, 2))(c8)
    u3 = tf.keras.layers.Concatenate()([u3, c4])
    c9 = conv_block(u3, 128)

    u4 = tf.keras.layers.UpSampling2D((2, 2))(c9)
    u4 = tf.keras.layers.Concatenate()([u4, c3])
    c10 = conv_block(u4, 64)

    u5 = tf.keras.layers.UpSampling2D((2, 2))(c10)
    u5 = tf.keras.layers.Concatenate()([u5, c2])
    c11 = conv_block(u5, 32)

    u6 = tf.keras.layers.UpSampling2D((2, 2))(c11)
    u6 = tf.keras.layers.Concatenate()([u6, c1])
    c12 = conv_block(u6, 16)

    # -------- Final Pixel-wise Classification --------
    outputs = tf.keras.layers.Conv2D(1, 1, activation="sigmoid")(c12)

    return tf.keras.Model(inputs, outputs)

# def conv_block(x, filters):
#     x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
#     x = tf.keras.layers.ReLU()(x)
#     x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
#     x = tf.keras.layers.ReLU()(x)
#     return x


# def conv_block(x, filters):
#     x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
#     x = tf.keras.layers.ReLU()(x)
#     x = tf.keras.layers.Conv2D(filters, 3, padding="same")(x)
#     x = tf.keras.layers.ReLU()(x)
#     return x


# def build_unet2d():
#     inputs = tf.keras.layers.Input((512, 512, 2))

#     # -------- Encoder --------
#     c1 = conv_block(inputs, 16)        # 512
#     p1 = tf.keras.layers.MaxPooling2D()(c1)

#     c2 = conv_block(p1, 32)            # 256
#     p2 = tf.keras.layers.MaxPooling2D()(c2)

#     c3 = conv_block(p2, 64)            # 128
#     p3 = tf.keras.layers.MaxPooling2D()(c3)

#     c4 = conv_block(p3, 128)           # 64
#     p4 = tf.keras.layers.MaxPooling2D()(c4)

#     c5 = conv_block(p4, 256)           # 32

#     # -------- Bottleneck --------
#     bn = conv_block(c5, 512)            # 32

#     # -------- Decoder --------
#     u1 = tf.keras.layers.UpSampling2D()(bn)     # 64
#     u1 = tf.keras.layers.Concatenate()([u1, c4])
#     c6 = conv_block(u1, 256)

#     u2 = tf.keras.layers.UpSampling2D()(c6)     # 128
#     u2 = tf.keras.layers.Concatenate()([u2, c3])
#     c7 = conv_block(u2, 128)

#     u3 = tf.keras.layers.UpSampling2D()(c7)     # 256
#     u3 = tf.keras.layers.Concatenate()([u3, c2])
#     c8 = conv_block(u3, 64)

#     u4 = tf.keras.layers.UpSampling2D()(c8)     # 512
#     u4 = tf.keras.layers.Concatenate()([u4, c1])
#     c9 = conv_block(u4, 32)

#     # -------- Output --------
#     outputs = tf.keras.layers.Conv2D(1, 1, activation="sigmoid")(c9)

#     return tf.keras.Model(inputs, outputs)


In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    return 1 - (2. * intersection + smooth) / (union + smooth)

In [ ]:
def binary_focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)

        return tf.reduce_mean(
            -alpha_t * tf.pow(1 - pt, gamma) * tf.math.log(pt)
        )
    return loss

In [ ]:
def focal_dice_loss(alpha=0.25, gamma=2.0):
    focal = binary_focal_loss(alpha, gamma)
    def loss(y_true, y_pred):
        return focal(y_true, y_pred) + dice_loss(y_true, y_pred)
    return loss

In [ ]:
class MeanIoUThreshold(tf.keras.metrics.MeanIoU):
    def __init__(self, threshold=0.4, name="mean_io_u", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.intersection = self.add_weight(name="intersection", initializer="zeros")
        self.union = self.add_weight(name="union", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)

        inter = tf.reduce_sum(y_true * y_pred)
        union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - inter

        self.intersection.assign_add(inter)
        self.union.assign_add(union)

    def result(self):
        return tf.math.divide_no_nan(self.intersection, self.union)

    def reset_states(self):
        self.intersection.assign(0.0)
        self.union.assign(0.0)

In [ ]:
model = build_unet2d()

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss=focal_loss(alpha=0.25, gamma=2.0),
#     metrics=[MeanIoUThreshold(num_classes=2, threshold=0.5)]
# )

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=focal_dice_loss(alpha=0.25, gamma=2.0),
    metrics=[MeanIoUThreshold(num_classes=2, threshold=0.4)]
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mean_io_u",
        mode="max",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_mean_io_u",
        mode="max",
        factor=0.3,
        patience=4,
        min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "stage2_unet_oil_segmentation.keras",
        monitor="val_mean_io_u",
        mode="max",
        save_best_only=True
    )
]

In [ ]:
img, mask = next(iter(train_ds))
print("Oil pixels:", tf.reduce_sum(mask).numpy())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.title("VV")
plt.imshow(img[0,:,:,0], cmap="gray")

plt.subplot(1,3,2)
plt.title("VH")
plt.imshow(img[0,:,:,1], cmap="gray")

plt.subplot(1,3,3)
plt.title("Mask")
plt.imshow(mask[0,:,:,0], cmap="gray")
plt.show()

In [ ]:
train_steps = max(len(train_i) // BATCH_SIZE, 1)
val_steps   = max(len(val_i) // BATCH_SIZE, 1)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=callbacks
)

In [ ]:
len(img), len(mask)

In [ ]:
import matplotlib.pyplot as plt

def visualize_sample(idx=0):
    img = read_image(val_i[idx])
    print(val_i[idx])
    gt  = read_mask(masks[idx])
    pred = model.predict(img[None])[0]
    pred = (pred > 0.5).astype(np.uint8)

    img = 0.3 * img[..., 0] + 0.7 * img[..., 1]

    print("Min:", pred.min())
    print("Max:", pred.max())
    print("Mean:", pred.mean())

    plt.figure(figsize=(15,5))
    plt.subplot(1,3,1); plt.title("SAR Image"); plt.imshow(img.squeeze(), cmap="gray")
    plt.subplot(1,3,2); plt.title("Ground Truth"); plt.imshow(gt.squeeze(), cmap="gray")
    plt.subplot(1,3,3); plt.title("Prediction"); plt.imshow(pred.squeeze(), cmap="gray")
    plt.show()

visualize_sample(1)


In [ ]:
model.save("/Users/parasningune/Desktop/Best_Models/Segmentation/model_segmentation.keras")

In [ ]:
model.save_weights("/Users/parasningune/Desktop/Best_Models/Segmentation/model_segmentation.weights.h5")

In [ ]:
import os
import numpy as np
import cv2
import tifffile as tiff
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 4 
EPOCHS = 50
LR = 1e-4
DATASET_DIR = "/Volumes/Windows8_OS/Dataset/Dataset-OG"

In [ ]:
def read_image(path):
    try:
        img = tiff.imread(path)
        # Handle (2,H,W) or (H,W,2)
        if img.ndim == 3 and img.shape[0] == 2:
            vv, vh = img[0], img[1]
        else:
            vv, vh = img[..., 0], img[..., 1]

        def sar_norm(ch):
            ch = ch.astype(np.float32)
            # Percentile clipping removes SAR outliers that squash contrast
            low, high = np.percentile(ch, (2, 98))
            ch = np.clip(ch, low, high)
            return (ch - low) / (high - low + 1e-8)

        vv = cv2.resize(sar_norm(vv), (IMG_SIZE, IMG_SIZE))
        vh = cv2.resize(sar_norm(vh), (IMG_SIZE, IMG_SIZE))
        return np.stack([vv, vh], axis=-1)
    except Exception as e:
        print(f"Error loading image {path}: {e}")
        return np.zeros((IMG_SIZE, IMG_SIZE, 2), dtype=np.float32)

In [ ]:
def read_mask(path):
    try:
        mask = tiff.imread(path)
        if mask.ndim > 2: mask = mask[..., 0]
        mask = (mask > 0).astype(np.float32)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        return mask[..., None]
    except Exception as e:
        print(f"Error loading mask {path}: {e}")
        return np.zeros((IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)

In [ ]:
def load_paths(root):
    imgs, masks = [], []
    for cls in ["Oil", "No_Oil"]:
        img_dir = os.path.join(root, "Images", cls)
        mask_dir = os.path.join(root, "Mask", cls)
        if not os.path.exists(img_dir): continue
        for img_name in os.listdir(img_dir):
            base, ext = os.path.splitext(img_name)
            img_path = os.path.join(img_dir, img_name)
            mask_path = os.path.join(mask_dir, base + "_segmentation" + ext)
            if os.path.exists(mask_path):
                imgs.append(img_path)
                masks.append(mask_path)
    return imgs, masks

In [ ]:
def augment(image, mask):
    # Geometric augmentations to help model generalize to different spill shapes
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)
    return image, mask

In [ ]:
def generator(imgs, masks):
    for i, m in zip(imgs, masks):
        yield read_image(i), read_mask(m)

In [ ]:
def make_dataset(imgs, masks, train=True):
    ds = tf.data.Dataset.from_generator(
        lambda: generator(imgs, masks),
        output_signature=(
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 2), tf.float32),
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 1), tf.float32)
        )
    )
    if train:
        ds = ds.shuffle(100).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
def conv_block(x, filters):
    # Added BatchNormalization and HeNormal initialization
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def build_unet2d():
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 2))
    
    # Encoder
    c1 = conv_block(inputs, 16)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    c2 = conv_block(p1, 32)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    c3 = conv_block(p2, 64)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    c4 = conv_block(p3, 128)
    p4 = layers.MaxPooling2D((2, 2))(c4)
    c5 = conv_block(p4, 256)
    p5 = layers.MaxPooling2D((2, 2))(c5)
    c6 = conv_block(p5, 512)
    p6 = layers.MaxPooling2D((2, 2))(c6)

    # Bottleneck
    bn = conv_block(p6, 1024)

    # Decoder
    def up_block(x, skip, filters):
        x = layers.UpSampling2D((2, 2))(x)
        x = layers.Concatenate()([x, skip])
        return conv_block(x, filters)

    d1 = up_block(bn, c6, 512)
    d2 = up_block(d1, c5, 256)
    d3 = up_block(d2, c4, 128)
    d4 = up_block(d3, c3, 64)
    d5 = up_block(d4, c2, 32)
    d6 = up_block(d5, c1, 16)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(d6)
    return models.Model(inputs, outputs)

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1. - (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

In [ ]:
def focal_dice_loss(y_true, y_pred):
    # Combination of focal and dice
    alpha, gamma = 0.75, 2.0 # Higher alpha for minority class (oil)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1. - 1e-7)
    focal = -y_true * alpha * tf.pow(1. - y_pred, gamma) * tf.math.log(y_pred) \
            - (1. - y_true) * (1. - alpha) * tf.pow(y_pred, gamma) * tf.math.log(1. - y_pred)
    return tf.reduce_mean(focal) + dice_loss(y_true, y_pred)

In [ ]:
class MeanIoUThreshold(tf.keras.metrics.Metric):
    def __init__(self, threshold=0.5, name="mean_iou", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.iou = self.add_weight(name="iou", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        intersection = tf.reduce_sum(y_true * y_pred)
        union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
        self.iou.assign_add(intersection / (union + 1e-7))
        self.count.assign_add(1.0)

    def result(self):
        return self.iou / self.count

In [ ]:
imgs, masks = load_paths(DATASET_DIR)
train_i, val_i, train_m, val_m = train_test_split(imgs, masks, test_size=0.2, random_state=42)

In [ ]:
train_ds = make_dataset(train_i, train_m, True)
val_ds = make_dataset(val_i, val_m, False)

In [ ]:
model = build_unet2d()
model.compile(optimizer=tf.keras.optimizers.Adam(LR), loss=focal_dice_loss, metrics=[MeanIoUThreshold()])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_mean_iou", mode="max", patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_mean_iou", mode="max", factor=0.2, patience=5),
    tf.keras.callbacks.ModelCheckpoint("best_oil_unet.keras", monitor="val_mean_iou", mode="max", save_best_only=True)
]

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=,
    steps_per_epoch=len(train_i)//BATCH_SIZE,
    validation_steps=len(val_i)//BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
import os
import numpy as np
import cv2
import tifffile as tiff
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 4          # safer for 512×512×2
EPOCHS = 50
LR = 1e-4

DATASET_DIR = "/Volumes/Windows8_OS/Dataset/Dataset-OG"


In [ ]:
def read_image(path):
    img = tiff.imread(path)

    # Handle (2,H,W) or (H,W,2)
    if img.ndim == 3 and img.shape[0] == 2:
        vv = img[0]
        vh = img[1]
    else:
        vv = img[..., 0]
        vh = img[..., 1]

    vv = vv.astype(np.float32)
    vh = vh.astype(np.float32)

    # Per-channel normalization
    vv = (vv - vv.min()) / (vv.max() - vv.min() + 1e-8)
    vh = (vh - vh.min()) / (vh.max() - vh.min() + 1e-8)

    vv = cv2.resize(vv, (IMG_SIZE, IMG_SIZE))
    vh = cv2.resize(vh, (IMG_SIZE, IMG_SIZE))

    img = np.stack([vv, vh], axis=-1)  # (512,512,2)
    return img


In [ ]:
def read_mask(path):
    mask = tiff.imread(path)

    if mask.ndim > 2:
        mask = mask[..., 0]

    mask = (mask > 0).astype(np.float32)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    return mask[..., None]   # (512,512,1)


In [ ]:
def load_paths(root):
    imgs, masks = [], []

    for cls in ["Oil"]:
        img_dir = os.path.join(root, "Images", cls)
        mask_dir = os.path.join(root, "Mask", cls)

        for img_name in os.listdir(img_dir):
            base, ext = os.path.splitext(img_name)

            img_path = os.path.join(img_dir, img_name)
            mask_path = os.path.join(mask_dir, base + "_segmentation" + ext)

            if not os.path.exists(mask_path):
                continue

            imgs.append(img_path)
            masks.append(mask_path)

    return imgs, masks


In [ ]:
imgs, masks = load_paths(DATASET_DIR)

train_i, val_i, train_m, val_m = train_test_split(
    imgs, masks, test_size=0.2, random_state=42
)


In [ ]:
def generator(imgs, masks):
    for i, m in zip(imgs, masks):
        try:
            yield read_image(i), read_mask(m)
        except:
            continue


In [ ]:
def make_dataset(imgs, masks, train=True):
    ds = tf.data.Dataset.from_generator(
        lambda: generator(imgs, masks),
        output_signature=(
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 2), tf.float32),
            tf.TensorSpec((IMG_SIZE, IMG_SIZE, 1), tf.float32)
        )
    )
    if train:
        ds = ds.shuffle(50)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
train_ds = make_dataset(train_i, train_m, train=True, repeat=True)
val_ds   = make_dataset(val_i, val_m, train=False, repeat=False)

In [ ]:
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same")(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(f, 3, padding="same")(x)
    x = layers.ReLU()(x)
    return x


In [ ]:
import keras_tuner as kt

def build_unet_tuned(hp):
    base_filters = hp.Choice("base_filters", [16, 32])
    bottleneck = hp.Choice("bottleneck", [256, 512])
    lr = hp.Choice("lr", [1e-3, 5e-4, 1e-4])
    pos_weight = hp.Choice("pos_weight", [3.0, 5.0, 7.0])

    def conv_block(x, f):
        x = layers.Conv2D(f, 3, padding="same")(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(f, 3, padding="same")(x)
        x = layers.ReLU()(x)
        return x

    i = layers.Input((IMG_SIZE, IMG_SIZE, 2))

    c1 = conv_block(i, base_filters); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, base_filters*2); p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, base_filters*4); p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, base_filters*8); p4 = layers.MaxPooling2D()(c4)
    c5 = conv_block(p4, bottleneck)

    u1 = layers.UpSampling2D()(c5)
    u1 = layers.Concatenate()([u1, c4])
    c6 = conv_block(u1, base_filters*8)

    u2 = layers.UpSampling2D()(c6)
    u2 = layers.Concatenate()([u2, c3])
    c7 = conv_block(u2, base_filters*4)

    u3 = layers.UpSampling2D()(c7)
    u3 = layers.Concatenate()([u3, c2])
    c8 = conv_block(u3, base_filters*2)

    u4 = layers.UpSampling2D()(c8)
    u4 = layers.Concatenate()([u4, c1])
    c9 = conv_block(u4, base_filters)

    o = layers.Conv2D(1, 1, activation="sigmoid")(c9)
    model = models.Model(i, o)

    model.compile(
        optimizer=Adam(lr),
        loss=weighted_bce(pos_weight),
        metrics=[iou_thresholded]
    )
    return model

In [ ]:
def weighted_bce(pos_weight=5.0):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        return tf.reduce_mean(
            - pos_weight * y_true * tf.math.log(y_pred)
            - (1 - y_true) * tf.math.log(1 - y_pred)
        )
    return loss


In [ ]:
def iou_thresholded(y_true, y_pred, thresh=0.7):
    y_pred = tf.cast(y_pred > thresh, tf.int32)
    y_true = tf.cast(y_true, tf.int32)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection

    intersection = tf.cast(intersection, tf.float32)
    union = tf.cast(union, tf.float32)

    return tf.where(
        tf.equal(union, 0.0),
        1.0,
        intersection / (union + 1e-6)
    )


In [ ]:
tuner = kt.RandomSearch(
    build_unet_tuned,
    objective=kt.Objective("val_iou_thresholded", direction="max"),
    max_trials=10,          # DO NOT go higher
    executions_per_trial=1,
    directory="tuner_logs",
    project_name="oil_unet_vv_vh"
)

In [ ]:
tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=25,   # short runs only
    callbacks=[
        EarlyStopping(
            monitor="val_iou_thresholded",
            patience=5,
            mode="max"
        )
    ]
)


In [ ]:
model = build_unet()

model.compile(
    optimizer=Adam(LR),
    loss=weighted_bce(pos_weight=5.0),
    metrics=[tf.keras.metrics.MeanIoU(num_classes=2)]
)

model.summary()


In [ ]:
callbacks = [
    EarlyStopping(monitor="val_mean_io_u", patience=10, mode="max", restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_mean_io_u", patience=4, factor=0.3, mode="max"),
    ModelCheckpoint("final_oil_unet_vv_vh.keras",
                    monitor="val_mean_io_u",
                    mode="max",
                    save_best_only=True)
]


In [ ]:
train_steps = max(len(train_i) // BATCH_SIZE, 1)
val_steps   = max(len(val_i) // BATCH_SIZE, 1)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)
